In [ ]:
FABRIC_SCOPE = 'https://api.fabric.microsoft.com'
POWERBI_SCOPE = 'https://analysis.windows.net/powerbi/api'
POWERBI_API = 'https://api.powerbi.com/v1.0/myorg'
FABRIC_API = 'https://api.fabric.microsoft.com/v1'

In [ ]:
import notebookutils
import requests
import pandas as pd


from deltalake import write_deltalake, DeltaTable

In [ ]:
def get_fabric_runtime_token(resource: str) -> str:
    """
    Get a user-scoped access token from the Fabric Notebook runtime for the given resource.
    The token inherits the signed-in user's permissions.
    """
    token = notebookutils.credentials.getToken(resource)
    if not token:
        raise RuntimeError("Failed to acquire token from notebookutils.credentials.getToken().")
    return token


def list_workspaces(access_token: str) -> list:
    headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {access_token}',
    }

    try:
        response = requests.request(
            method='GET',
            url=FABRIC_API + '/workspaces',
            headers=headers,
        )
    except Exception as e:
        raise RuntimeError("Failed to obtain capacities.")
    return response.json().get('value') 
    

In [ ]:
workspaces = list_workspaces(access_token=get_fabric_runtime_token(FABRIC_SCOPE))
df = pd.DataFrame(workspaces)
display(df) 


In [ ]:
write_deltalake(
    f'abfss://Sandbox@onelake.dfs.fabric.microsoft.com/LK_Sandbox.Lakehouse/Tables/Workspaces', 
    df, 
    mode='overwrite'
)
 